# Customer Churn Prediction & Retention Analytics

**Business Question:** Which customers are likely to leave, and what should the company do to retain them?

**Pipeline:**
- Data Cleaning → EDA → Statistical Analysis → Feature Engineering
- Model Building: Logistic Regression, Decision Tree, Random Forest
- Model Evaluation & Comparison
- Business Recommendations
- Power BI Export

**Dataset:** IBM Telco Customer Churn — 7,043 customer records

**Built by:** Rakesh | BCA — AI & Data Science

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("All libraries loaded successfully!")

## 2. Data Loading

In [ ]:
# Load the dataset
df = pd.read_csv('../data/telco_churn.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Total Customers: {len(df):,}")
print(f"Total Features: {df.shape[1]}")
df.head()

In [ ]:
# Dataset info
df.info()

In [ ]:
# Check column names and data types
print("Columns and Types:")
print("-" * 40)
for col in df.columns:
    print(f"  {col:25s} {str(df[col].dtype):10s} | {df[col].nunique()} unique values")

## 3. Data Cleaning

Key cleaning tasks:
1. Handle missing values
2. Fix data types (TotalCharges has blank strings)
3. Drop customerID (not a feature)
4. Convert SeniorCitizen to readable format

In [ ]:
# Check for missing values
print("Missing Values:")
print("-" * 40)
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No null values found")

# But TotalCharges has blank strings — check
print(f"\nBlank strings in TotalCharges: {(df['TotalCharges'] == ' ').sum()}")

In [ ]:
# Fix TotalCharges: convert to numeric, fill blanks with 0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Drop customerID — not a predictive feature
df.drop(columns=['customerID'], inplace=True)

# Convert SeniorCitizen from 0/1 to No/Yes for consistency
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

# Encode target: Churn Yes=1, No=0
df['Churn_Numeric'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Data cleaning complete!")
print(f"Final shape: {df.shape}")
df.dtypes

In [ ]:
# Check for duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")

# Statistical summary of numeric columns
df.describe().round(2)

## 4. Exploratory Data Analysis (EDA)

Let's explore the data to understand churn patterns and customer behavior.

### 4.1 Target Variable Distribution

In [ ]:
# Churn distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
churn_counts = df['Churn'].value_counts()
colors = ['#10b981', '#ef4444']
axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Customer Churn Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(churn_counts.values, labels=churn_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0, 0.05),
            textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Churn Rate', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

churn_rate = df['Churn_Numeric'].mean() * 100
print(f"Churn Rate: {churn_rate:.1f}%")
print(f"Retained: {churn_counts['No']:,} | Churned: {churn_counts['Yes']:,}")

### 4.2 Numerical Features Distribution

In [ ]:
# Distribution of numerical features by churn status
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, col in enumerate(numeric_cols):
    for label, color in [('No', '#10b981'), ('Yes', '#ef4444')]:
        subset = df[df['Churn'] == label][col]
        axes[i].hist(subset, bins=30, alpha=0.6, label=f'Churn={label}', color=color, edgecolor='white')
    axes[i].set_title(f'{col} by Churn Status', fontsize=13, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for numerical features
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, col in enumerate(numeric_cols):
    sns.boxplot(x='Churn', y=col, data=df, ax=axes[i],
                palette={'No': '#10b981', 'Yes': '#ef4444'})
    axes[i].set_title(f'{col} vs Churn', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Print mean values
print("Mean Values by Churn Status:")
print("-" * 50)
for col in numeric_cols:
    no_mean = df[df['Churn'] == 'No'][col].mean()
    yes_mean = df[df['Churn'] == 'Yes'][col].mean()
    print(f"  {col:20s} | Retained: {no_mean:8.2f} | Churned: {yes_mean:8.2f}")

### 4.3 Categorical Features vs Churn

In [ ]:
# Churn rate by categorical features
cat_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
            'InternetService', 'Contract', 'PaperlessBilling', 'PaymentMethod']

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate_by_cat = df.groupby(col)['Churn_Numeric'].mean() * 100
    bars = axes[i].bar(range(len(churn_rate_by_cat)), churn_rate_by_cat.values,
                       color='#0ea5e9', edgecolor='white', linewidth=1)
    axes[i].set_xticks(range(len(churn_rate_by_cat)))
    axes[i].set_xticklabels(churn_rate_by_cat.index, rotation=45, ha='right', fontsize=9)
    axes[i].set_title(f'Churn Rate by {col}', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].set_ylim(0, 60)
    
    # Add value labels
    for bar, val in zip(bars, churn_rate_by_cat.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

### 4.4 Correlation Analysis

In [ ]:
# Correlation heatmap for numeric features
numeric_df = df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_Numeric']].copy()
numeric_df.columns = ['Tenure', 'Monthly Charges', 'Total Charges', 'Churn']

corr_matrix = numeric_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.3f',
            square=True, linewidths=1, vmin=-1, vmax=1)
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key Correlations with Churn:")
print("-" * 40)
for col in ['Tenure', 'Monthly Charges', 'Total Charges']:
    corr_val = corr_matrix.loc['Churn', col]
    direction = "positive" if corr_val > 0 else "negative"
    print(f"  {col:20s} → {corr_val:+.3f} ({direction})")

## 5. Statistical Analysis

Using hypothesis tests to validate our EDA findings with statistical significance.

In [ ]:
# T-test: Compare mean tenure between churned and retained customers
churned_tenure = df[df['Churn'] == 'Yes']['tenure']
retained_tenure = df[df['Churn'] == 'No']['tenure']

t_stat, p_value = stats.ttest_ind(churned_tenure, retained_tenure)
print("T-Test: Tenure (Churned vs Retained)")
print("-" * 45)
print(f"  Churned mean tenure:  {churned_tenure.mean():.2f} months")
print(f"  Retained mean tenure: {retained_tenure.mean():.2f} months")
print(f"  T-statistic: {t_stat:.4f}")
print(f"  P-value: {p_value:.2e}")
print(f"  Significant? {'Yes ✓' if p_value < 0.05 else 'No ✗'}")
print(f"  → {'Churned customers have significantly shorter tenure' if p_value < 0.05 else 'No significant difference'}")

In [ ]:
# T-test: MonthlyCharges
churned_charges = df[df['Churn'] == 'Yes']['MonthlyCharges']
retained_charges = df[df['Churn'] == 'No']['MonthlyCharges']

t_stat, p_value = stats.ttest_ind(churned_charges, retained_charges)
print("T-Test: Monthly Charges (Churned vs Retained)")
print("-" * 45)
print(f"  Churned mean charges:  ${churned_charges.mean():.2f}")
print(f"  Retained mean charges: ${retained_charges.mean():.2f}")
print(f"  T-statistic: {t_stat:.4f}")
print(f"  P-value: {p_value:.2e}")
print(f"  Significant? {'Yes ✓' if p_value < 0.05 else 'No ✗'}")
print(f"  → {'Churned customers pay significantly higher monthly charges' if p_value < 0.05 else 'No significant difference'}")

In [ ]:
# Chi-Square Test: Contract type vs Churn
contingency = pd.crosstab(df['Contract'], df['Churn'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print("Chi-Square Test: Contract Type vs Churn")
print("-" * 45)
print(f"  Chi-square statistic: {chi2:.4f}")
print(f"  P-value: {p_value:.2e}")
print(f"  Degrees of freedom: {dof}")
print(f"  Significant? {'Yes ✓' if p_value < 0.05 else 'No ✗'}")
print(f"  → {'Contract type is significantly associated with churn' if p_value < 0.05 else 'No significant association'}")

print("\nChurn rates by Contract:")
for contract in df['Contract'].unique():
    rate = df[df['Contract'] == contract]['Churn_Numeric'].mean() * 100
    print(f"  {contract:20s} → {rate:.1f}% churn rate")

## 6. Feature Engineering

Preparing features for machine learning models:
1. Create tenure groups
2. Encode categorical variables
3. Scale numeric features
4. Train-test split

In [ ]:
# Create a clean copy for modeling
model_df = df.drop(columns=['Churn', 'Churn_Numeric']).copy()
target = df['Churn_Numeric'].copy()

# Create tenure groups
labels = [f"{i}-{i+11}" for i in range(1, 72, 12)]
model_df['tenure_group'] = pd.cut(model_df['tenure'].astype(int),
                                   bins=range(1, 80, 12),
                                   right=False, labels=labels)

# Drop raw tenure
model_df.drop(columns=['tenure'], inplace=True)

print("Features before encoding:")
print(f"  Shape: {model_df.shape}")
print(f"  Numeric: {model_df.select_dtypes(include=[np.number]).columns.tolist()}")
print(f"  Categorical: {model_df.select_dtypes(include=['object', 'category']).columns.tolist()}")

In [ ]:
# One-hot encode categorical features
numeric_cols = ['MonthlyCharges', 'TotalCharges']
categorical_cols = [c for c in model_df.columns if c not in numeric_cols]

X = pd.get_dummies(model_df, columns=categorical_cols, drop_first=False)

print(f"Features after encoding: {X.shape[1]} columns")
print(f"Target distribution: {target.value_counts().to_dict()}")

# Train-test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=0.2, random_state=42, stratify=target
)

print(f"\nTraining set: {X_train.shape[0]:,} samples")
print(f"Testing set:  {X_test.shape[0]:,} samples")
print(f"Churn rate in train: {y_train.mean()*100:.1f}%")
print(f"Churn rate in test:  {y_test.mean()*100:.1f}%")

In [ ]:
# Scale numeric features
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Feature scaling applied to:", numeric_cols)
print(f"Final feature matrix shape: {X_train.shape}")

## 7. Model Building

Training three classification models and comparing their performance.

### 7.1 Logistic Regression

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
lr_prob = lr_model.predict_proba(X_test)[:, 1]

print("LOGISTIC REGRESSION RESULTS")
print("=" * 45)
print(f"Accuracy:  {accuracy_score(y_test, lr_pred):.4f}")
print(f"Precision: {precision_score(y_test, lr_pred):.4f}")
print(f"Recall:    {recall_score(y_test, lr_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, lr_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, lr_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=['Retained', 'Churned']))

### 7.2 Decision Tree

In [ ]:
# Train Decision Tree
dt_model = DecisionTreeClassifier(max_depth=6, min_samples_split=10,
                                   min_samples_leaf=5, random_state=42)
dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)
dt_prob = dt_model.predict_proba(X_test)[:, 1]

print("DECISION TREE RESULTS")
print("=" * 45)
print(f"Accuracy:  {accuracy_score(y_test, dt_pred):.4f}")
print(f"Precision: {precision_score(y_test, dt_pred):.4f}")
print(f"Recall:    {recall_score(y_test, dt_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, dt_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, dt_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, dt_pred, target_names=['Retained', 'Churned']))

### 7.3 Random Forest

In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10,
                                   min_samples_split=5, min_samples_leaf=2,
                                   random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("RANDOM FOREST RESULTS")
print("=" * 45)
print(f"Accuracy:  {accuracy_score(y_test, rf_pred):.4f}")
print(f"Precision: {precision_score(y_test, rf_pred):.4f}")
print(f"Recall:    {recall_score(y_test, rf_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, rf_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, rf_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred, target_names=['Retained', 'Churned']))

## 8. Model Evaluation & Comparison

Comparing all three models side by side.

In [ ]:
# Model comparison table
models = {
    'Logistic Regression': (lr_pred, lr_prob),
    'Decision Tree': (dt_pred, dt_prob),
    'Random Forest': (rf_pred, rf_prob),
}

comparison = []
for name, (pred, prob) in models.items():
    comparison.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1 Score': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, prob),
    })

comparison_df = pd.DataFrame(comparison).set_index('Model')
comparison_df = comparison_df.round(4)

print("MODEL COMPARISON")
print("=" * 70)
print(comparison_df.to_string())

# Highlight the best model for each metric
print("\nBest model per metric:")
for col in comparison_df.columns:
    best = comparison_df[col].idxmax()
    print(f"  {col:12s} → {best} ({comparison_df.loc[best, col]:.4f})")

In [ ]:
# Comparison bar chart
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(comparison_df.columns))
width = 0.25
colors = ['#0ea5e9', '#8b5cf6', '#10b981']

for i, (model, row) in enumerate(comparison_df.iterrows()):
    bars = ax.bar(x + i * width, row.values, width, label=model,
                  color=colors[i], edgecolor='white', linewidth=1)
    for bar, val in zip(bars, row.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
               f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(comparison_df.columns, fontsize=11)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 7))

for name, (_, prob), color in zip(models.keys(), models.values(), colors):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=color, linewidth=2.5)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random Baseline')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — Model Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11, loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, (pred, _)) in enumerate(models.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Retained', 'Churned'],
                yticklabels=['Retained', 'Churned'])
    axes[i].set_title(f'{name}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Feature Importance Analysis

In [ ]:
# Random Forest Feature Importance (top 15)
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
top_15 = importances.nlargest(15)

plt.figure(figsize=(10, 7))
bars = plt.barh(range(len(top_15)), top_15.values, color='#0ea5e9', edgecolor='white')
plt.yticks(range(len(top_15)), [f.replace('_', ' ').title() for f in top_15.index], fontsize=11)
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 15 Features Driving Customer Churn (Random Forest)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()

for bar, val in zip(bars, top_15.values):
    plt.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("Top 5 Churn Drivers:")
for i, (feat, imp) in enumerate(top_15.head().items(), 1):
    print(f"  {i}. {feat.replace('_', ' ').title():40s} → {imp:.4f}")

In [ ]:
# Logistic Regression Coefficients (top 15 by absolute value)
lr_coefs = pd.Series(lr_model.coef_[0], index=X_train.columns)
top_coefs = lr_coefs.abs().nlargest(15)
top_coefs_signed = lr_coefs[top_coefs.index]

colors_coef = ['#ef4444' if v > 0 else '#10b981' for v in top_coefs_signed.values]

plt.figure(figsize=(10, 7))
plt.barh(range(len(top_coefs_signed)), top_coefs_signed.values, color=colors_coef, edgecolor='white')
plt.yticks(range(len(top_coefs_signed)),
           [f.replace('_', ' ').title() for f in top_coefs_signed.index], fontsize=11)
plt.xlabel('Coefficient (+ increases churn, - decreases churn)', fontsize=11)
plt.title('Top 15 Logistic Regression Coefficients', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Red = increases churn risk | Green = decreases churn risk")

## 10. Business Recommendations

Based on the analysis, here are actionable strategies for customer retention:

### Key Findings:
1. **Month-to-month contracts** have the highest churn rate (~42%) compared to one-year (~11%) and two-year (~3%) contracts
2. **Fiber optic internet** customers churn at ~42% vs DSL at ~19% — likely due to higher prices
3. **New customers** (tenure < 12 months) are the most vulnerable with the highest churn rates
4. **Higher monthly charges** strongly correlate with churn
5. **Customers without tech support and online security** are significantly more likely to churn
6. **Electronic check** payment method shows the highest churn rate (~45%)

### Retention Strategies:

| Strategy | Target Segment | Expected Impact |
|----------|---------------|-----------------|
| **Offer discounted annual contracts** | Month-to-month customers | Reduce churn from 42% by locking in commitment |
| **Bundle services at lower price** | Fiber optic without add-ons | Increase perceived value, reduce price sensitivity |
| **Onboarding program** | New customers (tenure < 6 months) | Reduce early churn through engagement |
| **Price optimization** | High monthly charge customers | Competitive pricing review to prevent price-driven exits |
| **Promote auto-pay methods** | Electronic check users | Switching to auto-pay reduces friction and churn |
| **Free trial of security & tech support** | Customers without these services | Add-on services increase switching cost |

### Prioritization:
- **Highest ROI**: Target month-to-month fiber optic customers with no add-on services — this is the highest-risk segment
- **Quick Win**: Encourage payment method migration from electronic check to automatic payment
- **Long-term**: Build loyalty programs that reward tenure milestones

## 11. Power BI Export

Exporting processed data for Power BI dashboard visualization.

In [ ]:
# Export 1: Clean dataset with churn predictions
export_df = pd.read_csv('../data/telco_churn.csv')
export_df['TotalCharges'] = pd.to_numeric(export_df['TotalCharges'], errors='coerce').fillna(0)

# Add tenure group
labels = [f"{i}-{i+11}" for i in range(1, 72, 12)]
export_df['tenure_group'] = pd.cut(export_df['tenure'].astype(int),
                                    bins=range(1, 80, 12),
                                    right=False, labels=labels)

# Add revenue category
export_df['revenue_category'] = pd.cut(export_df['MonthlyCharges'],
                                        bins=[0, 30, 60, 90, 150],
                                        labels=['Low', 'Medium', 'High', 'Premium'])

export_df.to_csv('../powerbi/churn_data_for_powerbi.csv', index=False)
print(f"Exported: churn_data_for_powerbi.csv ({len(export_df):,} records)")

# Export 2: Model comparison summary
comparison_df.to_csv('../powerbi/model_comparison.csv')
print(f"Exported: model_comparison.csv")

# Export 3: Feature importance
importance_export = importances.nlargest(20).reset_index()
importance_export.columns = ['Feature', 'Importance']
importance_export.to_csv('../powerbi/feature_importance.csv', index=False)
print(f"Exported: feature_importance.csv")

# Export 4: Churn rates by category
churn_summary = []
for col in ['Contract', 'InternetService', 'PaymentMethod', 'tenure_group', 'gender', 'SeniorCitizen']:
    if col in export_df.columns:
        rates = export_df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).round(1)
        for category, rate in rates.items():
            churn_summary.append({'Dimension': col, 'Category': str(category), 'Churn_Rate': rate})

churn_summary_df = pd.DataFrame(churn_summary)
churn_summary_df.to_csv('../powerbi/churn_rates_summary.csv', index=False)
print(f"Exported: churn_rates_summary.csv")

print("\nAll files exported to powerbi/ folder — ready for Power BI import!")

## 12. Conclusion

### Model Selection
- **Logistic Regression** offers the best interpretability with competitive performance
- **Random Forest** provides the highest accuracy and feature importance insights
- **Decision Tree** is the most explainable but has the lowest performance

### Best Model for Production: **Random Forest**
- Highest accuracy and AUC among the three models
- Feature importance helps explain predictions to business stakeholders
- Already deployed in the ChurnGuard web application

### Key Takeaway
Customer churn is driven primarily by **contract type**, **monthly charges**, **tenure**, and **service add-ons**. Proactive retention strategies targeting month-to-month, high-charge, early-tenure customers can significantly reduce churn.

---
*Analysis by Rakesh | BCA — AI & Data Science*